# Lab: Advanced Recursion + GUIs in Python
## IST1012 — Computer Programming II (Python)


**Description:** This lab builds on the previous recursion session. You will work with
recursion over *nested* structures (a simulated file system), use recursive **backtracking**
to crack a fixed-length PIN, and then wire a recursive function to a small **Tkinter GUI** so
a non-programmer analyst could run it. The theme throughout is practical security tooling.

**Estimated Time:** 45 minutes

---

## Learning Objectives

By the end of this lab, you will be able to:

1. Write a recursive function that traverses a nested (tree-like) data structure.
2. Implement recursive **backtracking** to explore a search space and stop at a goal.
3. Reason about depth, base cases, and what each recursive frame returns.
4. Connect a recursive function to a simple **Tkinter** GUI with a button and a result label.

---

## Part 1 — Warm-Up (5 min)

You have seen basic recursion already. Quick recall before we go deeper.

**Q1:** In one sentence, what is the difference between a *base case* and a *recursive case*?

**Q2:** The function below is meant to count how many items are in a nested list,
including items inside sub-lists. Trace it by hand for `count_items([1, [2, 3], [4, [5]]])`
**before** running it. What number do you expect?

In [2]:
from week03_dictionaries.lab import result

from week11_gui.lab import result_label


def count_items(data):
    total = 0
    for element in data:
        if isinstance(element, list):
            total = total + count_items(element)   # recursive case
        else:
            total = total + 1                      # base contribution
    return total

# Predict the output first, THEN run.
print(count_items([1, [2, 3], [4, [5]]]))


5


## Part 2 — Core Concepts (8 min)

### Concept A — Recursion over a tree

Real file systems are trees: a folder contains files *and* other folders. We model one with a
nested dictionary. To visit every node, a function calls itself on each child folder.

### Concept B — Recursive backtracking

Backtracking = "try a choice, recurse; if it leads nowhere, undo and try the next."
Brute-forcing a PIN of fixed length is exactly this: pick a digit for position 0, recurse to
fill position 1, and so on. When the string is complete, test it.

### Concept C — A recursive function behind a GUI

A GUI is just glue. The hard work stays in a normal recursive function; the GUI only collects
input (a text box), calls the function on a button click, and shows the result in a label.

Run the two examples below to anchor the ideas.

In [3]:
# Example: total size of every file in a nested folder tree
file_system = {
    "name": "root",
    "files": [("readme.txt", 20)],
    "folders": [
        {"name": "logs", "files": [("auth.log", 100), ("sys.log", 50)], "folders": []},
        {"name": "keys", "files": [("id_rsa", 5)], "folders": []}
    ]
}

def total_size(folder):
    size = 0
    for name, bytes_used in folder["files"]:
        size = size + bytes_used
    for sub in folder["folders"]:
        size = size + total_size(sub)   # recurse into each sub-folder
    return size

print("Total bytes:", total_size(file_system))


Total bytes: 175


In [4]:
# Example: build every 2-digit code recursively (a tiny backtracking preview)
def build_codes(prefix, length):
    if len(prefix) == length:      # base case: code is complete
        print(prefix)
        return
    for digit in "012":            # only 0,1,2 to keep output short
        build_codes(prefix + digit, length)

build_codes("", 2)


00
01
02
10
11
12
20
21
22


## Part 3 — Guided Exercises (25 min)

### Exercise 1 — Find a file in the tree *(warm)*

Using a nested folder dictionary like the one in Part 2, write a recursive function
`find_file(folder, target_name)` that returns `True` if a file with `target_name` exists
**anywhere** in the tree (including sub-folders), and `False` otherwise.

Requirements:
- Check the files in the current folder, then recurse into each sub-folder.
- Return as soon as you find it — do not keep searching unnecessarily.
- Do **not** use any GUI here; just a function and a couple of `print` tests.

Test it against the `file_system` from Part 2 (search for `"id_rsa"` and for `"nope.txt"`).

In [15]:
file_system = {
    "name": "root",
    "files": [("readme.txt", 20)],
    "folders": [
        {"name": "logs", "files": [("auth.log", 100), ("sys.log", 50)], "folders": []},
        {"name": "keys", "files": [("id_rsa", 5)], "folders": []}
    ]
}

def find_file(folder, target_name):
    # TODO: check this folder's files, then recurse into sub-folders
    for name, bytes_used in folder["files"]:
        if name == target_name:
            return True
    for sub in folder["folders"]:
        if find_file(sub, target_name):
            return True
    return False


# TODO: test with "id_rsa" (expect True) and "nope.txt" (expect False)
print("ID_RSA CHECK:", find_file(file_system, "id_rsa"))
print("NOPE.TXT CHECK:", find_file(file_system, "nope.txt"))

ID_RSA CHECK: True
NOPE.TXT CHECK: False


### Exercise 2 — Recursive PIN cracker with backtracking *(medium)*

A device is locked with a PIN of a known `length` using only digits `0-9`. You have a function
`check_pin(guess)` that returns `True` only for the correct PIN.

Write a recursive function `crack(prefix, length)` that uses backtracking to build candidate
PINs one digit at a time and returns the correct PIN string (or `None` if none works).

Requirements:
- Base case: when `len(prefix) == length`, test it with `check_pin`.
- Recursive case: try each digit `0`–`9`, append it, and recurse.
- Stop and return the answer the moment `check_pin` succeeds — do not test more PINs than needed.
- Use the provided `check_pin`; do not read `SECRET` directly in your logic.

The secret PIN below is `"73"`. Your function should discover it for `length = 2`.

In [19]:
SECRET = "73"   # pretend you do NOT know this value

def check_pin(guess):
    return guess == SECRET

def crack(prefix, length):
    # TODO: backtracking. Base case tests check_pin; recursive case tries 0-9.
    if len(prefix) == length:
        if check_pin(prefix):
            return prefix
        return None

    for digit in "0123456789":
        result = crack(prefix + digit, length)
        if result is not None:
            return result
    return None

# TODO: print(crack("", 2))  -> should print 73
print(crack("", 2))


73


### Exercise 3 — Put the cracker behind a GUI *(spicier)*

Now make Exercise 2 usable by a non-coder. Build a small **Tkinter** window with:

- An `Entry` box where the user types the **length** of the PIN.
- A **"Crack"** `Button`.
- A `Label` that shows the recovered PIN (or "not found").

When the button is clicked, read the length from the entry, call your recursive `crack`
function, and update the label with the result.

Requirements:
- Reuse your recursive `crack` from Exercise 2 — the recursion must stay in that function,
  not in the button handler.
- Convert the entry text to an integer safely (the user might type nonsense).
- The GUI code goes in the cell below.

> **Running note:** Tkinter opens a real window. Run this in a **local** Jupyter/Python install,
> not in a browser-only environment like Colab. If you are on Colab, describe in a comment what
> the handler would do instead of running it.

In [21]:
import tkinter as tk

SECRET = "73"
def check_pin(guess):
    return guess == SECRET

def crack(prefix, length):
    # TODO: paste your recursive solution from Exercise 2
    if len(prefix) == length:
        if check_pin(prefix):
            return prefix
        return None

    for digit in "0123456789":
        result = crack(prefix + digit, length)
        if result is not None:
            return result
    return None

def on_click():
    # TODO: read length from the entry, call crack, update result_label
    text = length_entry.get()
    try:
        length = int(text)
    except ValueError:
        result_label.config(text="Please enter a whole number.")
        return

    found = crack("",length)
    if found is None:
        result_label.config(text="PIN not found.")
    else:
        result_label.config(text="PIN is : " + found)

window = tk.Tk()
window.title("PIN Cracker")
# TODO: create an Entry, a Button (command=on_click), and a Label called result_label
# TODO: window.mainloop()

tk.Label(window, text="PIN Length:").pack()

length_entry = tk.Entry(window)
length_entry.pack()

tk.Button(window, text="Crack", command=on_click).pack()

result_label = tk.Label(window, text="")
result_label.pack()

window.mainloop()

---

**End of lab.** Save your notebook and submit per the README instructions.